# Player Tracking in Sports - Multi-View Tracking and 3D Reconstruction

## Imports

In [ ]:
from src.utils.video import open_video, get_frames, produce_tracking_output_video
from src.utils.visualization import show_images
from src.utils.annotations.download_annotations import download_annotations

from src.types.tracking import TrackingOutput, Frame_Detections

from src.detection.mog2_detection import run_mog2_detection, mog2_to_tracking_output
from src.tracking.yolo_tracking import run_yolo_tracking, yolo_to_tracking_output

import cv2
import time
import os
from pathlib import Path
from dotenv import load_dotenv
import logging
from ultralytics import YOLO

In [ ]:
CURRENT_CAMERA_ID = "cam_13"

## Path Definitions

In [ ]:
VIDEOS_DIR = "data/videos"
CAMERA_SETTINGS_DIR = "data/camera_settings"
MODELS_DIR = "models/"
ANNOTATIONS = "data/annotations/tracking_01"


# Define fundamental paths for each camera
CAMERAS = {
    "cam_13": {
        "video_path": f"{VIDEOS_DIR}/out13.mp4",
        "camera_settings": f"{CAMERA_SETTINGS_DIR}/cam13_settings.json",
    },
    "cam_2": {
        "video_path": f"{VIDEOS_DIR}/out2.mp4",
        "camera_settings": f"{CAMERA_SETTINGS_DIR}/cam2_settings.json",
    },
    "cam_4": {
        "video_path": f"{VIDEOS_DIR}/out4.mp4",
        "camera_settings": f"{CAMERA_SETTINGS_DIR}/cam4_settings.json",
    },
}

## Open Video and Read Frames

In [ ]:
# Currently just one camera
cap = open_video(CAMERAS[CURRENT_CAMERA_ID]["video_path"])
frames_color, _ = get_frames(cap, max_frames=150)  # Load only the first 150 frames for testing
fps = cap.get(cv2.CAP_PROP_FPS)

# Release the video capture object to free resources
cap.release()

show_images(frames_color)

## Background Subtraction with MOG2 for Player Detection

We use the MOG2 (Mixture of Gaussians v2) background subtractor to detect moving players in each frame.
As a background subtraction method, MOG2 resulted as a good choice for our application and through various experiments with other methods.

We further improved the detection results by applying pre-processing steps and post-processing steps to the frames and the foreground masks, respectively.


For **pre-processing**, we applied a illumination normalization technique called **CLAHE (Contrast Limited Adaptive Histogram Equalization)** to enhance the contrast of the frames, which helps in better foreground detection. Then to limit the noise generated from the shadow generated from the high reflective surfaces in the field, we removed from the detection mask the pixels that correspond to the hue values of the field.

For **post-processing**, we applied morphological operations such as opening and closing to remove noise and fill holes in the detected foreground masks, and then we applied contour detection to extract the bounding boxes of the detected players. We removed small contours that are likely to be noise by filtering based on contour area.

### Run MOG2 Detection on All Frames

We used as default parameters for the MOG2 background subtractor, the one that we found to be the best through fine-tuning and experimentation. 

This function outputs a list of bounding boxes for each frame, where each bounding box corresponds to a detected player. Each bounding box is represented as a tuple of (x, y, width, height).

In [ ]:
print(f"Processing {len(frames_color)} frames with MOG2 detection...")
start_time = time.time()

masks = run_mog2_detection(frames_color)

end_time = time.time()
print(f"MOG2 detection complete")
print(f"Processing time: {end_time - start_time:.2f} seconds")

show_images(masks)

### Convert Masks to Detection Output

Converts the blob masks obtained from MOG2 into bounding boxes for detected players. 

It's defined as a tracking output, without the tracking ID, which will be assigned in the tracking stage. 


In [ ]:
detection_output = mog2_to_tracking_output(
    masks,
    camera_id=CURRENT_CAMERA_ID,
    fps=fps,
)

print(f"Total detections: {sum(f.num_detections for f in detection_output.frames)} across {len(detection_output.frames)} frames")

### Inspect MOG2 Detections

We visualize the MOG2 blob detected by outputting the binary mask.
This allows us to inspect the quality of the foreground detection and to tune the parameters of the MOG2 background subtractor and the post-processing steps accordingly.

In [ ]:
print(f"First frame detections: {detection_output.frames[0].num_detections} objects detected")
print(f"First frame detection details:")
for i, detection in enumerate(detection_output.frames[0].detections[:5]):
    print(f"  Detection {i+1}: Class={detection.class_name}, Confidence={detection.confidence:.2f}, BBox={detection.bbox}")

In [ ]:
produce_tracking_output_video(frames_color, detection_output, f"results/detection/{CURRENT_CAMERA_ID}/mog2_detections.mp4")

## Using YOLOv11 for Player Detection

We could also use a deep learning-based object detection model like YOLOv11 to detect players in each frame. This can be done by loading a pre-trained YOLOv11 model and running inference on the frames to get bounding boxes for detected players.

In [ ]:
if not os.path.exists(MODELS_DIR):
    os.makedirs(MODELS_DIR)
model_path = os.path.join(MODELS_DIR, "yolo11m.pt")

model = YOLO(model_path)    # Load the YOLOv11m model

### Run YOLO Detection on All Frames

Use YOLOv8's built-in `.predict()` method to detect objects in each frame. The output will include bounding boxes, confidence scores, and class labels for each detected object.

**Output per frame:**
- Bounding boxes: `[x1, y1, x2, y2]` (pixel coordinates)
- Confidence scores: Detection confidence (0–1)
- Class labels: What was detected (e.g., "person", "sports_ball")

In [ ]:
print(f"Processing {len(frames_color)} frames with YOLO detection (player pass)...")
start_time = time.time()

# Pass 1 — players only at normal resolution.
# class_ids=[0] is COCO "person". After fine-tuning, swap to the new model's
# player/referee/goalkeeper IDs.
raw_player_results = run_yolo_tracking(
    model, frames_color,
    conf_threshold=0.4,
    inference_size=640,
    class_ids=[0],
)

end_time = time.time()
print(f"Player pass complete")
print(f"Total player-pass detections: {sum(len(r.boxes) for r in raw_player_results)}")
print(f"Processing time: {end_time - start_time:.2f} seconds")


### Higher-Resolution Pass — Ball Only

The football is a small object (~10–20 px in match footage) and the standard 640-px inference grid struggles to localize it. We therefore run a **second** inference pass at higher `imgsz` (1280) with a lower confidence threshold.

Critically, this pass is **class-filtered to the ball only**. Running high-resolution inference for `person` would also surface tiny bench players and spectators in the stands — exactly what we don't want.


In [ ]:
print(f"Processing {len(frames_color)} frames with YOLO detection (ball pass)...")
start_time = time.time()

# Pass 2 — ball only at high resolution.
# class_ids=[32] is COCO "sports ball". After fine-tuning, swap to the new model's ball ID.
# The low conf_threshold trades false positives for recall; tune it on your footage.
raw_ball_results = run_yolo_tracking(
    model, frames_color,
    conf_threshold=0.1,
    inference_size=1280,
    class_ids=[32],
)

end_time = time.time()
print(f"Ball pass complete")
print(f"Total ball-pass detections: {sum(len(r.boxes) for r in raw_ball_results)}")
print(f"Processing time: {end_time - start_time:.2f} seconds")


### Merge Player and Ball Detections

Combine both passes into a single `TrackingOutput`, frame by frame. After fine-tuning, only the `class_ids` lists in the two passes above need to change — the merge stays identical.


In [ ]:
player_out = yolo_to_tracking_output(
    raw_player_results, model,
    camera_id=CURRENT_CAMERA_ID, fps=25, source="yolo_v11m_pt",
)
ball_out = yolo_to_tracking_output(
    raw_ball_results, model,
    camera_id=CURRENT_CAMERA_ID, fps=25, source="yolo_v11m_pt",
)

detection_output = TrackingOutput(
    source=player_out.source,
    camera_id=player_out.camera_id,
    fps=player_out.fps,
    frames=[
        Frame_Detections(frame_index=p.frame_index, detections=p.detections + b.detections)
        for p, b in zip(player_out.frames, ball_out.frames)
    ],
)

print(f"Merged detections: {sum(f.num_detections for f in detection_output.frames)} across {len(detection_output.frames)} frames")


### Inspect Merged Detections

In [ ]:
print(f"First frame detections: {detection_output.frames[0].num_detections} objects detected")
print(f"First frame detection details:")
for i, detection in enumerate(detection_output.frames[0].detections[:5]):
    print(f"  Detection {i+1}: Class={detection.class_name}, Confidence={detection.confidence:.2f}, BBox={detection.bbox}")

# Quick visual check using YOLO's built-in .plot() on the player pass.
show_images(raw_player_results, is_yolo=True)

In [ ]:
produce_tracking_output_video(frames_color, ball_out, f"results/tracking/{CURRENT_CAMERA_ID}/tracking.mp4")

## Evaluation

In [ ]:
# Download annotations if not already present
if not any(Path(ANNOTATIONS).glob("*.json")):
    # Load the API key from the .env file and download annotations from Roboflow
    load_dotenv()
    api_key = os.environ["ANNOTATIONS_API_KEY"]
    download_annotations("tracking-rybv7", "tracking_01", 1, api_key, ANNOTATIONS)
else:
    print(f"Annotations already present in {ANNOTATIONS} - skipping download.")
